# LSI31003 Machine Learning in Molecular Biology
## Peptide classification group project

### Introduction:
In Assignment 3, you learned how pretrained transformer models could be used to learn embeddings for short peptide sequences, and how those embeddings could then be further used in downstream machine learning tasks. In assignment 3 we built a classifier to predict whether a given short peptide was of antimicrobial origin (AMP) vs non-Antimicrobial (nonAMP), based on the information contained within its sequence alone.

However, short peptide sequences have also other important functionality in living organisms than just Antimicrobial effects. In this group assignment, you will investigate peptides with multiple biological functionalities. 

Dataset comes from a curated short peptide collection described in (Xiao, B. *et al.*): **A comprehensive dataset of therapeutic peptides on multi-function property and structure information** (https://doi.org/10.1038/s41597-025-05528-1). Read the paper to find further details on how the peptides are classified (main and subcategories), as well as descriptions for the peptide metadata table.

For this task, we have calculated the ESM-2 embeddings for you in advance. All performed preprocessing steps are described in Peptide_project_preprocessing.ipynb. Furthermore, there is a second set of embeddings that is further derived from ESM-2 embeddings using supervised learning methods (known labels) with a deep learning framework (with combination of autoencoder-like reproducibility loss and prediction accuracy for the target label vector). You can use this set of embeddings as well, but if you are familiar with deep learning libraries such as pytorch and tensorflow, it's higly recommended that you try to learn suitable optimized embeddings yourself (but this is not a necessity)! The raw ESM-2 embeddings might also work on their own, but it's up to you to decide/ try out.

### Your task:
Your task is to built a classifier that predicts the peptide classes with as good accuracy as possible, mainly using the embedding representations as inputs (feel free to try other feature engineering methods as well if you like!). Apply dimensionsionality reduction methods if deemed appropriate. You should first make the classifier for the 15 main categories. After that, you can also try to predict the subcategories within each main peptide category. You should do model selection and show performance with confusion matrix visualizations and/or other general accuracy metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# your library imports
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import os
os.getcwd()

In [ ]:
# peptides with available metadata
meta = pd.read_csv("./data/Peptide_Metadata.csv", index_col = 0)
print(f"{meta.shape[0]} total peptides.")
meta.head(5)

In [ ]:
# Embeddings from ESM2-model (same pipeline as in Assignment 3)
embeddings_esm = pd.read_csv("./data/Embeddings_Esm2.csv", index_col = 0)
print(f"{embeddings_esm.shape[1]} dimensional peptide embeddings from ESM-2 model")
embeddings_esm.head()

In [ ]:
# Mystery embeddings: derivation from ESM-embeddings optimized with deeplearning methods and known labels.
embeddings_finetuned = pd.read_csv("./data/Embeddings_Mysteryengineered.csv", index_col = 0)
print(f"{embeddings_finetuned.shape[1]} dimensional peptide embeddings downstream engineered from ESM-2 model")
embeddings_finetuned.head()

**Labels -table columns:**  
`MAIN_category` one of the main 15 categories defined in the table (or "Multipurpose" if peptide belongs to multiple categories).  
`isSinglepurpose`: True if only 1 main category, False if MAIN_Category = "Multipurpose"  
`other columns`: Other peptide subtypes from the paper, corresponding to `Label encoding` -column of the meta table.

In [ ]:
labels = pd.read_csv("./data/Peptide_Targetlabels.csv", index_col = 0)
labels
labels.columns


## Remove Rare Subcategories
### Some subcategories had extremely low representation (<20 samples), which makes supervised learning unreliable. Therefore, we restricted the analysis to sufficiently represented subcategories to obtain meaningful performance metrics.

In [ ]:
# Count positives per label
label_counts = labels.drop(columns=["MAIN_Category", "isSinglepurpose"]).sum()

# Keep only labels with at least 20 samples
valid_labels = label_counts[label_counts >= 20].index

print("Number of subcategories kept:", len(valid_labels))

# New target matrix
y_filtered = labels[valid_labels].values

### 1. Identify subcategory columns

In [ ]:
# X = embeddings
X = embeddings_esm.values

# y = all subcategory columns (drop the first two columns)
y = labels.drop(columns=["MAIN_Category", "isSinglepurpose"]).values

### 2. Train–Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

### 3. Train Multi-Label Model

In [ ]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=3000
    )
)

model.fit(X_train, y_train)

import numpy as np

# Get probability predictions
y_proba = model.predict_proba(X_test)



### Apply custom Threshold

In [ ]:
threshold = 0.3
y_pred_custom = (y_proba >= threshold).astype(int)

### 6. Evaluate Model Performance

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_custom))

### Inspect Label Distribution

In [ ]:
label_counts = labels.drop(columns=["MAIN_Category", "isSinglepurpose"]).sum()
label_counts.sort_values(ascending=False).head(10)

# Final Model Evaluation and Interpretation

The subcategory prediction task was formulated as a multi-label classification problem, since a peptide can belong to multiple biological subcategories simultaneously.

We trained a One-vs-Rest Logistic Regression model using precomputed ESM-2 embeddings as input features.

Due to strong class imbalance across subcategories (e.g., Antibacterial and Antimicrobial dominating the dataset while several categories have very few samples), we adjusted the probability decision threshold from the default 0.5 to 0.3. This improved recall and overall F1-score without introducing excessive false positives.

Final Performance:

Micro F1-score: 0.63

Weighted F1-score: 0.61

Macro F1-score: 0.39

The model performs well on highly represented subcategories, while performance on rare categories remains limited due to insufficient training samples.